In [10]:
import torch

torch.cuda.empty_cache()

In [1]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

class SRImplicitDataset(Dataset):
    def __init__(self, img_dir, max_images=100):
        self.files = sorted([
            os.path.join(img_dir, f)
            for f in os.listdir(img_dir)
            if f.endswith(".png") or f.endswith(".jpg")
        ])[:max_images]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        hr = TF.to_tensor(img)

        h, w = hr.shape[1:]

        # HARD CLAMP HR (NO SKIP)
        h = max(2, h)
        w = max(2, w)
        hr = TF.resize(hr, (h, w), antialias=True)

        scale = random.uniform(1.5, 4.0)

        # SAFE LR (MIN = 2)
        lr_h = max(2, int(h / scale))
        lr_w = max(2, int(w / scale))

        lr = TF.resize(hr, (lr_h, lr_w), antialias=True)

        return lr, hr

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SRNOInspired(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, hidden, 3, padding=1),
            nn.ReLU()
        )

        self.mlp = nn.Sequential(
            nn.Linear(hidden + 2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 3)
        )

    def forward(self, lr, out_h, out_w):
        B, _, H_lr, W_lr = lr.shape

        # 🔥 FORCE SAFE INPUT SIZE
        if H_lr < 2 or W_lr < 2:
            lr = F.interpolate(lr, size=(max(2, H_lr), max(2, W_lr)), mode="bilinear", align_corners=False)

        feat = self.encoder(lr)

        B, C, H, W = feat.shape

        # FORCE SAFE FEATURE SIZE
        if H < 2 or W < 2:
            feat = F.interpolate(feat, size=(max(2, H), max(2, W)), mode="bilinear", align_corners=False)
            B, C, H, W = feat.shape

        # FORCE SAFE OUTPUT SIZE
        out_h = max(2, out_h)
        out_w = max(2, out_w)

        # coordinate grid
        y = torch.linspace(-1, 1, out_h, device=lr.device)
        x = torch.linspace(-1, 1, out_w, device=lr.device)
        yy, xx = torch.meshgrid(y, x, indexing="ij")

        grid = torch.stack((xx, yy), dim=-1)
        grid = grid.unsqueeze(0).repeat(B, 1, 1, 1)

        # 🔥 CRITICAL: safe grid_sample
        feat_up = F.grid_sample(
            feat,
            grid,
            mode='bilinear',
            align_corners=True,
            padding_mode='border'
        )

        feat_flat = feat_up.permute(0, 2, 3, 1).reshape(-1, C)
        coords_flat = grid.reshape(-1, 2)

        inp = torch.cat([feat_flat, coords_flat], dim=1)
        out = self.mlp(inp)

        out = out.view(B, out_h, out_w, 3).permute(0, 3, 1, 2)

        return out

In [3]:
#Load Dataset
import torch
import numpy as np
import cv2
import os
from ultralytics import YOLO
from PIL import Image
import torchvision.transforms.functional as TF

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load detector
detector = YOLO("C:/Users/Mardyson Justin/Thesis/yolov9c_visdrone_finetune15/weights/best.pt")

# Load SRNO
sr_model = SRNOInspired().to(device)
sr_model.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints5/180k_30e/best_model.pth",
    map_location=device
))
sr_model.eval()

SRNOInspired(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (mlp): Sequential(
    (0): Linear(in_features=258, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=3, bias=True)
  )
)

In [3]:
# Load Dataset FRCNN
import torch
import numpy as np
import cv2
import os
from PIL import Image
import torchvision.transforms.functional as TF

from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# LOAD FASTER R-CNN DETECTOR
# -----------------------------
NUM_CLASSES = 11  # adjust if needed (background + classes)

detector = fasterrcnn_mobilenet_v3_large_fpn(weights=None)

in_features = detector.roi_heads.box_predictor.cls_score.in_features
detector.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

detector.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/FCNN3/checkpoints/best_model.pth",
    map_location=device
))

detector.to(device)
detector.eval()

# -----------------------------
# LOAD SR MODEL
# -----------------------------
sr_model = SRNOInspired().to(device)

sr_model.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints5/180k_30e/best_model.pth",
    map_location=device
))

sr_model.eval()

SRNOInspired(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (mlp): Sequential(
    (0): Linear(in_features=258, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=3, bias=True)
  )
)

In [12]:
def frcnn_detect(image_np, model, device, thresh=0.05):
    img = TF.to_tensor(image_np).to(device)

    with torch.no_grad():
        out = model([img])[0]

    boxes = out["boxes"].cpu().numpy()
    scores = out["scores"].cpu().numpy()
    labels = out["labels"].cpu().numpy()

    keep_boxes = []
    keep_scores = []
    keep_labels = []

    for b, s, l in zip(boxes, scores, labels):

        if s < thresh:
            continue

        if l == 0:
            continue

        keep_boxes.append(b)
        keep_scores.append(s)
        keep_labels.append(l - 1)  # your mapping

    return keep_boxes, keep_scores, keep_labels

In [4]:
from torch import autocast
from PIL import Image
import numpy as np
import cv2
import torchvision.transforms.functional as TF
import torch

def detect_sr_icro(
    image_path, model, device,
    tau_low=0.3,
    tau_high=0.5,
    min_alpha=1.5,
    alpha_max=4,
    epsilon=0.09,
    max_iter=6,
    alpha_blend_default=0.5,
    max_roi_size=128,
    save_output=False,
    output_path=None
):

    def size_category(w, h):
        area = w*h
        if area < 1024:
            return "tiny"
        elif area < 9216:
            return "small"
        else:
            return "medium"

    # Alpha caps per category
    alpha_caps = {"tiny": 3.0, "small": 2.0, "medium": 1.8}

    image = Image.open(image_path).convert("RGB")
    current_img = np.array(image)
    metrics_log = []

    num_objects_estimate = 1000
    object_scales = [1.0] * num_objects_estimate
    object_iterations = [1] * num_objects_estimate

    for iteration in range(max_iter):
        boxes, confs, labels = frcnn_detect(current_img, sr_model, device)

        if boxes is None or len(boxes) == 0:
            print("No detections.")
            break

        confs_before = boxes.conf.cpu().numpy()
        avg_conf_before = np.mean(confs_before)
        updated_regions = 0
        scales_used = []

        tiny_objects_present = False

        for idx, box in enumerate(boxes):
            conf = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            roi = current_img[y1:y2, x1:x2]

            H, W = roi.shape[:2] if roi.size > 0 else (0, 0)
            category = size_category(W, H)

            # Skip SR if already high confidence
            if conf >= tau_high or roi.size == 0 or H > max_roi_size or W > max_roi_size:
                object_scales[idx] = max(object_scales[idx], min_alpha)
                object_iterations[idx] = max(object_iterations[idx], iteration + 1)
                continue

            if category == "tiny":
                tiny_objects_present = True
                max_obj_iter = max_iter
            elif category == "small":
                max_obj_iter = max_iter - 1
            else:
                max_obj_iter = 1

            if iteration + 1 > max_obj_iter:
                object_scales[idx] = max(object_scales[idx], min_alpha)
                object_iterations[idx] = max(object_iterations[idx], iteration + 1)
                continue

            # Adaptive alpha
            if conf <= tau_low:
                base_alpha = alpha_max
            else:
                ratio = (tau_high - conf) / (tau_high - tau_low)
                base_alpha = min_alpha + (alpha_max - min_alpha) * (ratio ** 1.5)

            # Size and iteration boosts
            size_boost = 1.0 if category == "tiny" else 0.75 if category == "small" else 0.5
            iteration_boost = 1.0 + 0.1 * iteration
            alpha = base_alpha * size_boost * iteration_boost

            # Cap alpha per category
            alpha = min(alpha, alpha_caps[category])
            scales_used.append(alpha)

            # ROI tensor
            roi_pil = Image.fromarray(roi)
            model_device = next(sr_model.parameters()).device
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(model_device)
            if next(sr_model.parameters()).dtype == torch.float16:
                lr_tensor = lr_tensor.half()

            target_h = int(H * alpha)
            target_w = int(W * alpha)

            with torch.no_grad(), autocast(device_type='cuda', enabled=(lr_tensor.dtype == torch.float16)):
                torch.cuda.empty_cache()
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_img = TF.to_pil_image(sr.squeeze(0).clamp(0, 1).cpu())
            sr_np = np.array(sr_img)
            sr_np_resized = cv2.resize(sr_np, (W, H), interpolation=cv2.INTER_CUBIC)

            # Confidence-weighted blending with safe max
            delta_conf_roi = tau_high - conf
            alpha_blend_roi = alpha_blend_default * (delta_conf_roi / (tau_high - tau_low))
            alpha_blend_roi = np.clip(alpha_blend_roi, 0.3, 0.7)

            if category == "tiny":
                alpha_blend_roi *= 1.1
            elif category == "small":
                alpha_blend_roi *= 1.05
            alpha_blend_roi = np.clip(alpha_blend_roi, 0.3, 0.8)

            mask = cv2.GaussianBlur(np.ones_like(roi, dtype=np.float32), (11,11), 0)
            blended_roi = (alpha_blend_roi * (mask * sr_np_resized) + (1 - alpha_blend_roi) * roi).astype(np.uint8)
            current_img[y1:y2, x1:x2] = blended_roi
            updated_regions += 1

            # Update tracking
            object_scales[idx] = max(object_scales[idx], alpha)
            object_iterations[idx] = max(object_iterations[idx], iteration + 1)

        # Post-iteration metrics
        results_after = detector(current_img)[0]
        boxes_after = results_after.boxes
        if boxes_after is None or len(boxes_after) == 0:
            break

        confs_after = boxes_after.conf.cpu().numpy()
        avg_conf_after = np.mean(confs_after)
        delta_conf = avg_conf_after - avg_conf_before
        avg_scale = np.mean(scales_used) if scales_used else 1.0

        metrics_log.append({
            "iteration": iteration + 1,
            "avg_scaling_factor": float(avg_scale),
            "regions_updated": updated_regions,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf)
        })

        print(f"\nIteration {iteration+1}")
        print(f"Avg Scaling Factor   : {avg_scale:.2f}")
        print(f"Updated Regions      : {updated_regions}")
        print(f"Avg Confidence Before: {avg_conf_before:.4f}")
        print(f"Avg Confidence After : {avg_conf_after:.4f}")
        print(f"Delta Confidence     : {delta_conf:.4f}")

        # Convergence checks
        delta_conf_effective = delta_conf * 0.5 if tiny_objects_present else delta_conf
        if abs(delta_conf_effective) < epsilon and not tiny_objects_present:
            print("Converged.")
            break
        if avg_conf_after >= tau_high:
            print("Reached Stable High Confidence.")
            break

    if save_output and output_path is not None:
        cv2.imwrite(output_path, current_img)
    
    current_img = cv2.GaussianBlur(current_img, (3,3), 0)

    return current_img, metrics_log, object_scales[:len(boxes)], object_iterations[:len(boxes)]

In [8]:
#working for simsar
def detect_sr_icro_fixed(
    image_path,
    model, device,
    tau_low=0.3,
    tau_high=0.5,
    fixed_scales=[2, 3, 4],
    epsilon=0.05,
    max_iter=4,
    alpha_blend_default=0.5,
    max_roi_size=128,
    save_output=False,
    output_path=None
):

    def size_category(w, h):
        area = w*h
        if area < 1024:
            return "tiny"
        elif area < 9216:
            return "small"
        else:
            return "medium"

    image = Image.open(image_path).convert("RGB")
    current_img = np.array(image)
    metrics_log = []

    num_objects_estimate = 1000
    object_scales = [1.0] * num_objects_estimate
    object_iterations = [1] * num_objects_estimate

    for iteration in range(max_iter):
        img_tensor = TF.to_tensor(current_img).unsqueeze(0).to(device)

        with torch.no_grad():
            results = detector(img_tensor)[0]
        boxes = results.boxes

        if boxes is None or len(boxes) == 0:
            print("No detections.")
            break

        confs_before = boxes.conf.cpu().numpy()
        avg_conf_before = np.mean(confs_before)
        updated_regions = 0
        scales_used = []

        for idx, box in enumerate(boxes):
            conf = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            roi = current_img[y1:y2, x1:x2]

            H, W = roi.shape[:2] if roi.size > 0 else (0,0)
            category = size_category(W, H)

            if roi.size == 0 or H > max_roi_size or W > max_roi_size:
                continue

            # ------------------------
            # Dynamic scale selection
            # ------------------------
            if conf >= tau_high:
                alpha = 1.0
            else:
                if category == "tiny":
                    alpha = fixed_scales[min(iteration, len(fixed_scales)-1)]
                elif category == "small":
                    alpha = fixed_scales[min(iteration, 1)]
                else:
                    alpha = fixed_scales[0]

            scales_used.append(alpha)

            # ROI tensor
            roi_pil = Image.fromarray(roi)
            model_device = next(sr_model.parameters()).device
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(model_device)

            if next(sr_model.parameters()).dtype == torch.float16:
                lr_tensor = lr_tensor.half()

            target_h = int(H * alpha)
            target_w = int(W * alpha)

            with torch.no_grad(), autocast(device_type='cuda', enabled=(lr_tensor.dtype == torch.float16)):
                torch.cuda.empty_cache()
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_img = TF.to_pil_image(sr.squeeze(0).clamp(0, 1).cpu())
            sr_np = np.array(sr_img)

            # keep cubic
            sr_np_resized = cv2.resize(sr_np, (W, H), interpolation=cv2.INTER_CUBIC)

            # ✅ FIXED: stronger lower bound to prevent artifacts
            delta_conf_roi = max(tau_high - conf, 0)
            alpha_blend_roi = alpha_blend_default * (delta_conf_roi / (tau_high - tau_low))
            alpha_blend_roi = np.clip(alpha_blend_roi, 0.4, 0.7)

            blended_roi = (alpha_blend_roi * sr_np_resized + (1 - alpha_blend_roi) * roi).astype(np.uint8)
            current_img[y1:y2, x1:x2] = blended_roi
            updated_regions += 1

            object_scales[idx] = max(object_scales[idx], alpha)
            object_iterations[idx] = max(object_iterations[idx], iteration + 1)

        # ------------------------
        # Confidence check
        # ------------------------
        results_after = detector(current_img)[0]
        boxes_after = results_after.boxes
        if boxes_after is None or len(boxes_after) == 0:
            break

        confs_after = boxes_after.conf.cpu().numpy()
        avg_conf_after = np.mean(confs_after)
        delta_conf = avg_conf_after - avg_conf_before
        avg_scale = np.mean(scales_used) if scales_used else 1.0

        metrics_log.append({
            "iteration": iteration + 1,
            "avg_scaling_factor": float(avg_scale),
            "regions_updated": updated_regions,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf)
        })

        print(f"\nIteration {iteration+1}")
        print(f"Avg Scaling Factor   : {avg_scale:.2f}")
        print(f"Updated Regions      : {updated_regions}")
        print(f"Avg Confidence Before: {avg_conf_before:.4f}")
        print(f"Avg Confidence After : {avg_conf_after:.4f}")
        print(f"Delta Confidence     : {delta_conf:.4f}")

        if abs(delta_conf) < epsilon:
            print("Converged.")
            break
        if avg_conf_after >= tau_high:
            print("Reached Stable High Confidence.")
            break

    if save_output and output_path is not None:
        cv2.imwrite(output_path, current_img)

    return current_img, metrics_log, object_scales[:len(boxes)], object_iterations[:len(boxes)]

In [4]:
from torch import autocast
from PIL import Image
import numpy as np
import cv2
import torchvision.transforms.functional as TF
import torch

def detect_sr_icro_from_array(
    input_img,
    tau_low=0.3,
    tau_high=0.6,
    min_alpha=1.5,
    alpha_max=4,
    alpha_blend_default=0.5,
    epsilon=0.09,
    max_iter=6,
    max_roi_size=128,
    save_output=False,
    output_path=None
):
    # Copy input image
    current_img = np.array(input_img).astype(np.uint8).copy()
    metrics_log = []

    num_objects_estimate = 1000
    object_scales = [1.0] * num_objects_estimate
    object_iterations = [1] * num_objects_estimate

    def size_category(w, h):
        area = w * h
        if area < 1024:
            return "tiny"
        elif area < 9216:
            return "small"
        else:
            return "medium"

    for iteration in range(max_iter):
        # Detect objects
        results = detector(current_img)[0]
        boxes = results.boxes
        if boxes is None or len(boxes) == 0:
            break

        confs_before = boxes.conf.cpu().numpy()
        avg_conf_before = np.mean(confs_before)
        updated_regions = 0
        scales_used = []

        for idx, box in enumerate(boxes):
            conf = float(box.conf)

            if conf >= tau_high:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            roi = current_img[y1:y2, x1:x2]
            if roi.size == 0:
                continue

            H, W = roi.shape[:2]
            if H > max_roi_size or W > max_roi_size:
                continue

            # ---- Adaptive scaling alpha ----
            if conf <= tau_low:
                alpha = alpha_max
            else:
                alpha = 1 + 5.0 * (tau_high - conf)
            alpha = max(min_alpha, min(alpha, alpha_max))
            scales_used.append(alpha)

            # Convert ROI to tensor for SR
            roi_pil = Image.fromarray(roi)
            model_device = next(sr_model.parameters()).device
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(model_device)
            if next(sr_model.parameters()).dtype == torch.float16:
                lr_tensor = lr_tensor.half()

            target_h = int(H * alpha)
            target_w = int(W * alpha)

            # ---- SR Inference ----
            with torch.no_grad(), autocast(device_type='cuda', enabled=(lr_tensor.dtype == torch.float16)):
                torch.cuda.empty_cache()
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_img = TF.to_pil_image(sr.squeeze(0).clamp(0, 1).cpu())
            sr_np = np.array(sr_img)
            sr_np_resized = cv2.resize(sr_np, (W, H), interpolation=cv2.INTER_CUBIC)

            # ---- Alpha blending ----
            delta_conf_roi = tau_high - conf
            alpha_blend_roi = alpha_blend_default * (delta_conf_roi / (tau_high - tau_low))

            # Boost for small/tiny objects
            category = size_category(W, H)
            if category == "tiny":
                alpha_blend_roi *= 1.1
            elif category == "small":
                alpha_blend_roi *= 1.05

            alpha_blend_roi = np.clip(alpha_blend_roi, 0.3, 0.8)

            blended_roi = (alpha_blend_roi * sr_np_resized + (1 - alpha_blend_roi) * roi).astype(np.uint8)
            current_img[y1:y2, x1:x2] = blended_roi
            updated_regions += 1

            # Update per-object tracking
            object_scales[idx] = max(object_scales[idx], alpha)
            object_iterations[idx] = max(object_iterations[idx], iteration + 1)

        # ---- Re-detection for convergence ----
        results_after = detector(current_img)[0]
        boxes_after = results_after.boxes
        if boxes_after is None or len(boxes_after) == 0:
            break

        confs_after = boxes_after.conf.cpu().numpy()
        avg_conf_after = np.mean(confs_after)
        delta_conf = avg_conf_after - avg_conf_before
        avg_scale = np.mean(scales_used) if scales_used else 1.0

        metrics_log.append({
            "iteration": iteration + 1,
            "avg_scaling_factor": float(avg_scale),
            "regions_updated": updated_regions,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf)
        })

        # Convergence check
        if abs(delta_conf) < epsilon or avg_conf_after >= tau_high:
            break

    # Save output if requested
    if save_output and output_path is not None:
        cv2.imwrite(output_path, current_img)

    return current_img, metrics_log, object_scales[:len(boxes)], object_iterations[:len(boxes)]

In [5]:
def detect_sr_icro_fixed_from_array(
    input_img,
    tau_low=0.3,
    tau_high=0.5,
    fixed_scales=[2,3,4],
    epsilon=0.05,
    max_iter=4,
    alpha_blend_default=0.5,
    max_roi_size=128,
    save_output=False,
    output_path=None
):

    def size_category(w,h):
        area = w*h
        if area < 1024:
            return "tiny"
        elif area < 9216:
            return "small"
        else:
            return "medium"

    current_img = np.array(input_img).astype(np.uint8).copy()
    metrics_log = []

    object_scales = []
    object_iterations = []

    for iteration in range(max_iter):

        results = detector(current_img)[0]
        boxes = results.boxes

        if boxes is None or len(boxes) == 0:
            break

        confs_before = boxes.conf.cpu().numpy()
        avg_conf_before = np.mean(confs_before)

        updated_regions = 0
        scales_used = []

        for idx, box in enumerate(boxes):

            conf = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            roi = current_img[y1:y2, x1:x2]

            H, W = roi.shape[:2] if roi.size > 0 else (0,0)
            category = size_category(W, H)

            # ---- per-object iteration logic ----
            if category == "tiny":
                max_obj_iter = max_iter
            else:
                max_obj_iter = 1

            for obj_iter in range(max_obj_iter):

                # Skip conditions
                if conf >= tau_high or roi.size == 0 or H > max_roi_size or W > max_roi_size:
                    object_scales.append(1.0)
                    object_iterations.append(iteration + 1)
                    continue

                # ---- scaling logic ----
                if category == "tiny":
                    alpha = fixed_scales[min(obj_iter, len(fixed_scales)-1)]
                else:
                    alpha = fixed_scales[0]

                scales_used.append(alpha)

                # ---- SR ----
                roi_pil = Image.fromarray(roi)
                model_device = next(sr_model.parameters()).device
                lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(model_device)

                if next(sr_model.parameters()).dtype == torch.float16:
                    lr_tensor = lr_tensor.half()

                target_h = int(H * alpha)
                target_w = int(W * alpha)

                with torch.no_grad(), autocast(device_type='cuda', enabled=(lr_tensor.dtype == torch.float16)):
                    torch.cuda.empty_cache()
                    sr = sr_model(lr_tensor, target_h, target_w)

                sr_img = TF.to_pil_image(sr.squeeze(0).clamp(0,1).cpu())
                sr_np = np.array(sr_img)

                sr_np_resized = cv2.resize(sr_np, (W, H), interpolation=cv2.INTER_CUBIC)

                # BLENDING 
                delta_conf_roi = tau_high - conf
                alpha_blend_roi = alpha_blend_default * (delta_conf_roi / (tau_high - tau_low))
                alpha_blend_roi = np.clip(alpha_blend_roi, 0.3, 0.8)

                blended_roi = (
                    alpha_blend_roi * sr_np_resized +
                    (1 - alpha_blend_roi) * roi
                ).astype(np.uint8)

                current_img[y1:y2, x1:x2] = blended_roi
                updated_regions += 1

                # Track
                object_scales.append(alpha)
                object_iterations.append(iteration + 1)

        # ---- Re-detection ----
        results_after = detector(current_img)[0]
        boxes_after = results_after.boxes

        if boxes_after is None or len(boxes_after) == 0:
            break

        confs_after = boxes_after.conf.cpu().numpy()
        avg_conf_after = np.mean(confs_after)

        delta_conf = avg_conf_after - avg_conf_before
        avg_scale = np.mean(scales_used) if scales_used else 1.0

        metrics_log.append({
            "iteration": iteration + 1,
            "avg_scaling_factor": float(avg_scale),
            "regions_updated": updated_regions,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf)
        })

        # ---- stopping conditions ----
        if abs(delta_conf) < epsilon:
            break
        if avg_conf_after >= tau_high:
            break

    if save_output and output_path is not None:
        cv2.imwrite(output_path, current_img)

    return current_img, metrics_log, object_scales, object_iterations

In [6]:
#wurking fixed
def detect_sr_icro_fixed_from_array(
    input_img,
    tau_low=0.3,
    tau_high=0.5,
    fixed_scales=[2, 3, 4],
    epsilon=0.09,
    max_iter=4,
    max_roi_size=128,
    save_output=False,
    output_path=None
):

    def size_category(w, h):
        area = w*h
        if area < 1024:
            return "tiny"
        elif area < 9216:
            return "small"
        else:
            return "medium"

    current_img = np.array(input_img).astype(np.uint8).copy()

    metrics_log = []

    num_objects_estimate = 1000
    object_scales = [1.0] * num_objects_estimate
    object_iterations = [1] * num_objects_estimate

    for iteration in range(max_iter):

        results = detector(current_img)[0]
        boxes = results.boxes

        if boxes is None or len(boxes) == 0:
            break

        confs_before = boxes.conf.cpu().numpy()
        avg_conf_before = np.mean(confs_before)

        updated_regions = 0
        scales_used = []

        for idx, box in enumerate(boxes):

            conf = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            roi = current_img[y1:y2, x1:x2]

            H, W = roi.shape[:2] if roi.size > 0 else (0,0)
            category = size_category(W, H)

            if roi.size == 0 or H > max_roi_size or W > max_roi_size:
                continue

            # ---- OLD SCALING LOGIC ----
            if conf >= tau_high:
                continue
            else:
                if category == "tiny":
                    alpha = fixed_scales[min(iteration, len(fixed_scales)-1)]
                elif category == "small":
                    alpha = fixed_scales[min(iteration, 1)]
                else:
                    alpha = fixed_scales[0]

            scales_used.append(alpha)

            roi_pil = Image.fromarray(roi)
            model_device = next(sr_model.parameters()).device
            lr_tensor = TF.to_tensor(roi_pil).unsqueeze(0).to(model_device)

            if next(sr_model.parameters()).dtype == torch.float16:
                lr_tensor = lr_tensor.half()

            target_h = int(H * alpha)
            target_w = int(W * alpha)

            # ---- PURE SR (NO BLENDING) ----
            with torch.no_grad(), autocast(device_type='cuda', enabled=(lr_tensor.dtype == torch.float16)):
                torch.cuda.empty_cache()
                sr = sr_model(lr_tensor, target_h, target_w)

            sr_img = TF.to_pil_image(sr.squeeze(0).clamp(0,1).cpu())
            sr_np = np.array(sr_img)

            sr_np_resized = cv2.resize(sr_np, (W, H), interpolation=cv2.INTER_CUBIC)

            
            current_img[y1:y2, x1:x2] = sr_np_resized
            updated_regions += 1

            object_scales[idx] = max(object_scales[idx], alpha)
            object_iterations[idx] = max(object_iterations[idx], iteration + 1)

        # ---- Re-detection ----
        results_after = detector(current_img)[0]
        boxes_after = results_after.boxes

        if boxes_after is None or len(boxes_after) == 0:
            break

        confs_after = boxes_after.conf.cpu().numpy()
        avg_conf_after = np.mean(confs_after)
        delta_conf = avg_conf_after - avg_conf_before

        avg_scale = np.mean(scales_used) if scales_used else 1.0

        metrics_log.append({
            "iteration": iteration + 1,
            "avg_scaling_factor": float(avg_scale),
            "regions_updated": updated_regions,
            "avg_conf_before": float(avg_conf_before),
            "avg_conf_after": float(avg_conf_after),
            "delta_conf": float(delta_conf)
        })

        if abs(delta_conf) < epsilon:
            break
        if avg_conf_after >= tau_high:
            break

    if save_output and output_path is not None:
        cv2.imwrite(output_path, current_img)

    return current_img, metrics_log, object_scales[:len(boxes)], object_iterations[:len(boxes)]

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

sr_model = sr_model.to(device)
sr_model.eval()

SRNOInspired(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (mlp): Sequential(
    (0): Linear(in_features=258, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=3, bias=True)
  )
)

In [ ]:
#woking2
# SIMULATION
import cv2
import lpips
import os
import torch
import torchvision.transforms.functional as TF
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tkinter import Tk, filedialog
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

device = "cuda" if torch.cuda.is_available() else "cpu"

# LPIPS perceptual similarity
lpips_model = lpips.LPIPS(net='alex').to(device)
lpips_model.eval()

output_dir = "C:/Users/Mardyson Justin/Thesis/OutputsFinal/"
os.makedirs(output_dir, exist_ok=True)

MIN_SIZE = 32
OBJECT_FILTER = "medium"   

def resize_if_needed(img, min_size=MIN_SIZE):
    h, w = img.shape[:2]
    scale = max(min_size / h, min_size / w)
    if scale > 1:
        new_h, new_w = int(h * scale), int(w * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
    return img

def size_category(w,h):
    area = w*h
    if area < 1024:
        return "tiny"
    elif area < 9216:
        return "small"
    else:
        return "medium"
    
    
class_names = {
    0: 'pedestrian',
    1: 'person',
    2: 'bicycle',
    3: 'car',
    4: 'van',
    5: 'truck',
    6: 'tricycle',
    7: 'awning-tricycle',
    8: 'bus',
    9: 'motor'
}

def safe_crop(img, x1, y1, x2, y2):
    h, w = img.shape[:2]

    x1 = max(0, min(w - 1, x1))
    x2 = max(0, min(w, x2))
    y1 = max(0, min(h - 1, y1))
    y2 = max(0, min(h, y2))

    if x2 <= x1 or y2 <= y1:
        return None

    crop = img[y1:y2, x1:x2]

    if crop.size == 0:
        return None

    return crop

# -----------------------------
# FILE PICKER
# -----------------------------
def select_image():
    root = Tk()
    root.withdraw()
    file_path = filedialog.askopenfilename(
        title="Select an image",
        filetypes=[("Image Files", "*.jpg *.png *.jpeg")]
    )
    return file_path

# -----------------------------
# DRAW DETECTIONS
# -----------------------------
def draw_detections(image, results, title=""):
    img = image.copy()
    if results.boxes is None:
        return img

    boxes = results.boxes.xyxy.cpu().numpy()
    confs = results.boxes.conf.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    for box, conf, cls in zip(boxes, confs, classes):
        x1, y1, x2, y2 = map(int, box)

        # Compute size category
        width = x2 - x1
        height = y2 - y1
        size_class = size_category(width, height)

        if OBJECT_FILTER is not None and size_class != OBJECT_FILTER:
            continue

        # Label with class, confidence, and size
        label = f"{class_names[int(cls)]}: {conf:.2f}, {size_class}"

        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.45
        thickness = 1
        text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]

        text_x1 = x1
        text_y1 = y1 - text_size[1] - 4
        if text_y1 < 0:
            text_y1 = y1 + text_size[1] + 4
        text_x2 = text_x1 + text_size[0] + 4
        text_y2 = text_y1 + text_size[1] + 4

        overlay = img.copy()
        cv2.rectangle(overlay, (text_x1, text_y1), (text_x2, text_y2), (0, 255, 0), cv2.FILLED)
        img = cv2.addWeighted(overlay, 0.5, img, 0.5, 0)
        cv2.putText(img, label, (text_x1 + 2, text_y2 - 2),
                    font, font_scale, (0, 0, 0), thickness)

    if title:
        cv2.putText(img, title, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (0, 0, 255), 2)
    return img

# -----------------------------
# DRAW METRICS PER OBJECT
# -----------------------------
def draw_metrics(img, box, text_lines, model_label=None):
    x1, y1, x2, y2 = box
    cv2.rectangle(img, (x1,y1), (x2,y2), (255, 0, 0), 2)
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.4
    thickness = 1

    for i, line in enumerate(text_lines):
        text_size = cv2.getTextSize(line, font, font_scale, thickness)[0]
        tx = x1
        ty = y1 - (len(text_lines)-i)*(text_size[1]+4)
        if ty < 0:
            ty = y2 + i*(text_size[1]+4) + 4

        overlay = img.copy()
        cv2.rectangle(overlay,
                      (tx, ty),
                      (tx + text_size[0] + 4, ty + text_size[1] + 4),
                      (255, 0, 0),
                      -1)
        img[:] = cv2.addWeighted(overlay, 0.5, img, 0.5, 0)
        cv2.putText(img, line, (tx + 2, ty + text_size[1] + 2),
                    font, font_scale, (255,255,255), thickness)

    if model_label:
        label_size = cv2.getTextSize(model_label, font, 1, 2)[0]
        overlay = img.copy()
        cv2.rectangle(overlay,
                      (5,5),
                      (5 + label_size[0] + 6, 5 + label_size[1] + 6),
                      (0, 0, 255),
                      -1)
        img[:] = cv2.addWeighted(overlay, 0.5, img, 0.5, 0)
        cv2.putText(img, model_label, (8, 5 + label_size[1]),
                    font, 0.7, (255,255,255), 2)
        
# -----------------------------
# DRAW BOTTOM-RIGHT LABELS
# -----------------------------
def draw_bottom_label(img, text, font_scale=0.5, color=(0,0,255)):
    font = cv2.FONT_HERSHEY_SIMPLEX
    thickness = 1
    text_size = cv2.getTextSize(text, font, font_scale, thickness)[0]
    h, w = img.shape[:2]
    x = w - text_size[0] - 10
    y = h - 10
    overlay = img.copy()
    cv2.rectangle(overlay, (x-2, y-text_size[1]-2), (x + text_size[0]+2, y+2), color, -1)
    img[:] = cv2.addWeighted(overlay, 0.6, img, 0.5, 0)
    cv2.putText(img, text, (x, y), font, font_scale, (255,255,255), thickness)


def draw_avg_metrics(img, psnr, ssim, lp, model_label):
    draw_bottom_label(img,
        f"{model_label} Avg -> PSNR:{psnr:.1f} SSIM:{ssim:.2f} LPIPS:{lp:.3f}")
    


# -----------------------------
# METRICS FUNCTIONS
# -----------------------------
def compute_metrics(ref, test):
    if ref.size == 0 or test.size == 0:
        return 0, 0, 0

    h, w = ref.shape[:2]
    if h < 16 or w < 16:
        return 0, 0, 0

    # --- PSNR / SSIM (unchanged) ---
    ref_gray = cv2.cvtColor(ref, cv2.COLOR_RGB2GRAY)
    test_gray = cv2.cvtColor(test, cv2.COLOR_RGB2GRAY)

    psnr = peak_signal_noise_ratio(ref_gray, test_gray)
    ssim = structural_similarity(ref_gray, test_gray)

    # --- Adaptive resizing to minimum size (for small ROIs) ---

    ref = resize_if_needed(ref)
    test = resize_if_needed(test)

    # =============================
    # ✅ LPIPS FAIRNESS STEP
    # =============================
    LPIPS_SIZE = 128

    if LPIPS_SIZE is not None:
        ref_lp = cv2.resize(ref, (LPIPS_SIZE, LPIPS_SIZE), interpolation=cv2.INTER_AREA)
        test_lp = cv2.resize(test, (LPIPS_SIZE, LPIPS_SIZE), interpolation=cv2.INTER_AREA)
    else:
        ref_lp = ref.copy()
        test_lp = test.copy()

    # Optional: smaller blur (or remove it)
    ref_lp = cv2.GaussianBlur(ref_lp, (1,1), 0)
    test_lp = cv2.GaussianBlur(test_lp, (1,1), 0)

    # Convert to tensor
    t1 = TF.to_tensor(ref_lp).unsqueeze(0).to(device) * 2 - 1
    t2 = TF.to_tensor(test_lp).unsqueeze(0).to(device) * 2 - 1

    with torch.no_grad():
        lp = lpips_model(t1, t2).item()

    return psnr, ssim, lp


def variance_of_laplacian(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

# -----------------------------
# MAIN PIPELINE
# -----------------------------
def run_simulation():

    img_path = select_image()
    if not img_path:
        print("No image selected.")
        return
    print("Selected:", img_path)

    hr_img = Image.open(img_path).convert("RGB")
    hr_np = np.array(hr_img)
    H, W = hr_np.shape[:2]

    # -----------------------------
    # HR detection for ground truth normalization
    # -----------------------------
    hr_results = detector(hr_np, imgsz=640)[0]
    if hr_results.boxes is not None:
        gt_count = len(hr_results.boxes)
    else:
        gt_count = 0

    # -----------------------------
    # REALISTIC LR SIMULATION
    # -----------------------------
    scale = 4
    scale_m = 2
    scale_mm = 1.5

    lr = cv2.resize(hr_np, (W//scale, H//scale), interpolation=cv2.INTER_AREA)
    lr = cv2.GaussianBlur(lr, (3,3), 0.5)
    noise = np.random.normal(0, 6, lr.shape).astype(np.float32)
    lr = np.clip(lr.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    _, encimg = cv2.imencode('.jpg', lr, [int(cv2.IMWRITE_JPEG_QUALITY), 60])
    lr = cv2.imdecode(encimg, 1)

    lr_m = cv2.resize(hr_np, (int(W//scale_m), int(H//scale_m)), interpolation=cv2.INTER_AREA)
    lr_m = cv2.GaussianBlur(lr_m, (3,3), 0.5)
    noise_m = np.random.normal(0, 5, lr_m.shape).astype(np.float32)
    lr_m = np.clip(lr_m.astype(np.float32) + noise_m, 0, 255).astype(np.uint8)
    _, encimg_m = cv2.imencode('.jpg', lr_m, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
    lr_m = cv2.imdecode(encimg_m, 1)

    lr_mm = cv2.resize(hr_np, (int(W//scale_mm), int(H//scale_mm)), interpolation=cv2.INTER_AREA)
    lr_mm = cv2.GaussianBlur(lr_mm, (3,3), 0.5)
    noise_mm = np.random.normal(0, 3, lr_mm.shape).astype(np.float32)
    lr_mm = np.clip(lr_mm.astype(np.float32) + noise_mm, 0, 255).astype(np.uint8)
    _, encimg_mm = cv2.imencode('.jpg', lr_mm, [int(cv2.IMWRITE_JPEG_QUALITY), 70])
    lr_mm = cv2.imdecode(encimg_mm, 1)

    # -----------------------------
    # YOLO
    # -----------------------------
    yolo_results = detector(lr, imgsz=640)[0]
    yolo_img = draw_detections(lr, yolo_results, "YOLO")

    scale_factor = 4  # 2x bigger, adjust as needed
    yolo_img = cv2.resize(yolo_img, (W*scale_factor, H*scale_factor), interpolation=cv2.INTER_LINEAR)

    conf_list = []
    if yolo_results.boxes is not None:
        boxes = yolo_results.boxes.xyxy.cpu().numpy()
        confs = yolo_results.boxes.conf.cpu().numpy()

        for box, conf in zip(boxes, confs):
            x1, y1, x2, y2 = map(int, box)
            w, h = x2 - x1, y2 - y1
            size = size_category(w, h)

            if OBJECT_FILTER is None or size == OBJECT_FILTER:
                conf_list.append(conf)

    conf_list = np.array(conf_list)
    filtered_count = len(conf_list)
    avg_conf = conf_list.mean() if len(conf_list) > 0 else 0
    draw_bottom_label(yolo_img, f"Avg Conf: {avg_conf:.3f}", font_scale=1*scale_factor)

    # -----------------------------
    # SR
    # -----------------------------
    lr_tensor = TF.to_tensor(lr_m).unsqueeze(0).to(device)
    with torch.no_grad():
        sr = sr_model(lr_tensor, H, W)

    sr_np = np.array(TF.to_pil_image(sr.squeeze().clamp(0,1).cpu()))
    sr_results = detector(sr_np, imgsz=640)[0]
    sr_img = draw_detections(sr_np, sr_results, "YOLO with SR")

    conf_sr_list = []

    if sr_results.boxes is not None:
        boxes = sr_results.boxes.xyxy.cpu().numpy()
        confs = sr_results.boxes.conf.cpu().numpy()

        for box, conf in zip(boxes, confs):
            x1, y1, x2, y2 = map(int, box)
            w, h = x2 - x1, y2 - y1
            size = size_category(w, h)

            if OBJECT_FILTER is None or size == OBJECT_FILTER:
                conf_sr_list.append(conf)

    conf_sr_list = np.array(conf_sr_list)
    conf_sr_sum = conf_sr_list.sum() if len(conf_sr_list) > 0 else 0
    avg_conf_sr = conf_sr_list.mean() if len(conf_sr_list) > 0 else 0
    draw_bottom_label(sr_img, f"Avg Conf: {avg_conf_sr:.3f}", font_scale=1.2)

    # -----------------------------
    # Fixed ICRO
    # -----------------------------
    sr_fixed, _, _, _ = detect_sr_icro_fixed_from_array(lr_mm)
    fixed_results = detector(sr_fixed, imgsz=640)[0]
    fixed_img = draw_detections(sr_fixed, fixed_results, "YOLO with SR + PARTIAL ICRO")

    conf_fx_list = []

    if sr_results.boxes is not None:
        boxes = fixed_results.boxes.xyxy.cpu().numpy()
        confs = fixed_results.boxes.conf.cpu().numpy()

        for box, conf in zip(boxes, confs):
            x1, y1, x2, y2 = map(int, box)
            w, h = x2 - x1, y2 - y1
            size = size_category(w, h)

            if OBJECT_FILTER is None or size == OBJECT_FILTER:
                conf_fx_list.append(conf)

    conf_fx_list = np.array(conf_fx_list)
    conf_fx_sum = conf_fx_list.sum() if len(conf_fx_list) > 0 else 0
    avg_conf_fx = conf_fx_list.mean() if len(conf_fx_list) > 0 else 0
    draw_bottom_label(fixed_img, f"Avg Conf: {avg_conf_fx:.3f}", font_scale=1.5)

    # -----------------------------
    # Adaptive ICRO
    # -----------------------------
    sr_adapt = lr_mm.copy()
    if yolo_results.boxes is not None:
        for box in yolo_results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])
            crop_lr = safe_crop(lr_mm, x1, y1, x2, y2)
            if crop_lr is None:
                continue
            sr_crop, _, _, _ = detect_sr_icro_from_array(crop_lr)
            sr_adapt[y1:y2, x1:x2] = sr_crop

    adapt_results = detector(sr_adapt, imgsz=640)[0]
    adapt_img = draw_detections(sr_adapt, adapt_results, "YOLO with SR + FULL ICRO")

    conf_ad_list = []

    if sr_results.boxes is not None:
        boxes = adapt_results.boxes.xyxy.cpu().numpy()
        confs = adapt_results.boxes.conf.cpu().numpy()

        for box, conf in zip(boxes, confs):
            x1, y1, x2, y2 = map(int, box)
            w, h = x2 - x1, y2 - y1
            size = size_category(w, h)

            if OBJECT_FILTER is None or size == OBJECT_FILTER:
                conf_ad_list.append(conf)

    conf_ad_list = np.array(conf_ad_list)
    conf_ad_sum = conf_ad_list.sum() if len(conf_ad_list) > 0 else 0
    avg_conf_ad = conf_ad_list.mean() if len(conf_ad_list) > 0 else 0
    draw_bottom_label(adapt_img, f"Avg Conf: {avg_conf_ad:.3f}", font_scale=1.5)

    # -----------------------------
    # METRICS
    # -----------------------------
    metrics_sr_img = sr_np.copy()
    metrics_fixed_img = sr_fixed.copy()
    metrics_adapt_img = sr_adapt.copy()

    sr_fixed = cv2.resize(sr_fixed, (W, H), interpolation=cv2.INTER_CUBIC)
    sr_adapt = cv2.resize(sr_adapt, (W, H), interpolation=cv2.INTER_CUBIC)

    metrics_fixed_img = cv2.resize(metrics_fixed_img, (W, H))
    metrics_adapt_img = cv2.resize(metrics_adapt_img, (W, H))

    psnr_sr_w = ssim_sr_w = lp_sr_w = 0
    psnr_fx_w = ssim_fx_w = lp_fx_w = 0
    psnr_ad_w = ssim_ad_w = lp_ad_w = 0
    total_weight = 0

    if hr_results.boxes is not None:
        for i, box in enumerate(hr_results.boxes):

            x1, y1, x2, y2 = map(int, box.xyxy.cpu().numpy()[0])

            w, h = x2 - x1, y2 - y1
            size = size_category(w, h)
            if OBJECT_FILTER is not None and size != OBJECT_FILTER:
                continue

            crop_hr = safe_crop(hr_np, x1, y1, x2, y2)
            crop_sr = safe_crop(sr_np, x1, y1, x2, y2)
            crop_fixed = safe_crop(sr_fixed, x1, y1, x2, y2)
            crop_adapt = safe_crop(sr_adapt, x1, y1, x2, y2)

            if crop_hr is None or crop_sr is None or crop_fixed is None or crop_adapt is None:
                continue

            target_h, target_w = crop_hr.shape[:2]
            crop_sr = cv2.resize(crop_sr, (target_w, target_h))
            crop_fixed = cv2.resize(crop_fixed, (target_w, target_h))
            crop_adapt = cv2.resize(crop_adapt, (target_w, target_h))

            crop_hr_resized = resize_if_needed(crop_hr)
            crop_sr_resized = resize_if_needed(crop_sr)
            crop_fixed_resized = resize_if_needed(crop_fixed)
            crop_adapt_resized = resize_if_needed(crop_adapt)

            psnr_sr, ssim_sr, lp_sr = compute_metrics(crop_hr_resized, crop_sr_resized)
            psnr_fx, ssim_fx, lp_fx = compute_metrics(crop_hr_resized, crop_fixed_resized)
            psnr_ad, ssim_ad, lp_ad = compute_metrics(crop_hr_resized, crop_adapt_resized)

            # ✅ WEIGHTED ACCUMULATION
            area = (x2 - x1) * (y2 - y1)
            weight = max(area, 1)

            psnr_sr_w += psnr_sr * weight
            ssim_sr_w += ssim_sr * weight
            lp_sr_w += lp_sr * weight

            psnr_fx_w += psnr_fx * weight
            ssim_fx_w += ssim_fx * weight
            lp_fx_w += lp_fx * weight

            psnr_ad_w += psnr_ad * weight
            ssim_ad_w += ssim_ad * weight
            lp_ad_w += lp_ad * weight

            total_weight += weight

            draw_metrics(metrics_sr_img, [x1,y1,x2,y2],
                [f"PSNR:{psnr_sr:.1f}", f"SSIM:{ssim_sr:.2f}", f"LPIPS:{lp_sr:.3f}"], "YOLO with SRNO")

            draw_metrics(metrics_fixed_img, [x1,y1,x2,y2],
                [f"PSNR:{psnr_fx:.1f}", f"SSIM:{ssim_fx:.2f}", f"LPIPS:{lp_fx:.3f}"], "YOLO with SR + Partial")

            draw_metrics(metrics_adapt_img, [x1,y1,x2,y2],
                [f"PSNR:{psnr_ad:.1f}", f"SSIM:{ssim_ad:.2f}", f"LPIPS:{lp_ad:.3f}"], "YOLO with SR + Full ICRO")

    # ✅ COMPUTE AVERAGES
    if total_weight > 0:
        avg_psnr_sr = psnr_sr_w / total_weight
        avg_ssim_sr = ssim_sr_w / total_weight
        avg_lp_sr = lp_sr_w / total_weight

        avg_psnr_fx = psnr_fx_w / total_weight
        avg_ssim_fx = ssim_fx_w / total_weight
        avg_lp_fx = lp_fx_w / total_weight

        avg_psnr_ad = psnr_ad_w / total_weight
        avg_ssim_ad = ssim_ad_w / total_weight
        avg_lp_ad = lp_ad_w / total_weight
    else:
        avg_psnr_sr = avg_ssim_sr = avg_lp_sr = 0
        avg_psnr_fx = avg_ssim_fx = avg_lp_fx = 0
        avg_psnr_ad = avg_ssim_ad = avg_lp_ad = 0

    # ✅ DRAW AVERAGES
    draw_avg_metrics(metrics_sr_img, avg_psnr_sr, avg_ssim_sr, avg_lp_sr, "SR")
    draw_avg_metrics(metrics_fixed_img, avg_psnr_fx, avg_ssim_fx, avg_lp_fx, "Partial ICRO")
    draw_avg_metrics(metrics_adapt_img, avg_psnr_ad, avg_ssim_ad, avg_lp_ad, "Full ICRO")

    # -----------------------------
    # SAVE (UNCHANGED)
    # -----------------------------
    cv2.imwrite(os.path.join(output_dir, "output_yolo.jpg"), cv2.cvtColor(yolo_img, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(output_dir, "output_sr.jpg"), cv2.cvtColor(sr_img, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(output_dir, "output_fixed.jpg"), cv2.cvtColor(fixed_img, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(output_dir, "output_adaptive.jpg"), cv2.cvtColor(adapt_img, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(output_dir, "metrics_sr.jpg"), cv2.cvtColor(metrics_sr_img, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(output_dir, "metrics_fixed.jpg"), cv2.cvtColor(metrics_fixed_img, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(output_dir, "metrics_adapt.jpg"), cv2.cvtColor(metrics_adapt_img, cv2.COLOR_RGB2BGR))

    # ✅ SIDE-BY-SIDE METRICS IMAGE
    def resize_same(img): return cv2.resize(img, (W, H))
    m1, m2, m3 = resize_same(metrics_sr_img), resize_same(metrics_fixed_img), resize_same(metrics_adapt_img)

    def resize_same2(img): 
        return cv2.resize(img, (W, H))

    yolo_r  = resize_same2(yolo_img)
    sr_r    = resize_same2(sr_img)
    fixed_r = resize_same2(fixed_img)
    adapt_r = resize_same2(adapt_img)

    top = np.hstack((yolo_r, sr_r))
    bottom = np.hstack((fixed_r, adapt_r))

    grid = np.vstack((top, bottom))

    cv2.imwrite(
        os.path.join(output_dir, "comparison_2x2.jpg"),
        cv2.cvtColor(grid, cv2.COLOR_RGB2BGR)
    )
    metrics_grid = np.hstack((m1, m2, m3))
    cv2.imwrite(os.path.join(output_dir, "metrics_comparison.jpg"),
                cv2.cvtColor(metrics_grid, cv2.COLOR_RGB2BGR))
    
    
run_simulation() #workeng

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\lpips\weights\v0.1\alex.pth
Selected: C:/Users/Mardyson Justin/OneDrive/Pictures/Screenshots/Screenshot 2026-04-06 221612.png

0: 384x640 6 pedestrians, 1 person, 1 bicycle, 11 cars, 1 van, 2 trucks, 1 tricycle, 1 awning-tricycle, 1 motor, 253.4ms
Speed: 4.4ms preprocess, 253.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 van, 1 truck, 23.5ms
Speed: 1.8ms preprocess, 23.5ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 pedestrian, 3 cars, 2 vans, 1 truck, 1 motor, 21.3ms
Speed: 1.2ms preprocess, 21.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x672 8 pedestrians, 10 cars, 2 vans, 1 truck, 1 tricycle, 1 awning-tricycle, 1 motor, 66.6ms
Speed: 1.8ms preprocess, 66.6ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 672)

0: 38